# SeisMambaKAN — Colab

Production notebook. Three cells:

1. **Bootstrap**: mount Drive, clone GitHub repo, change cwd.
2. **Setup**: install deps + sync processed data from Drive.
3. **Run**: train / eval / infer.

Open this file directly via:
`https://colab.research.google.com/github/huseyinokanozturk/SeisMambaKAN/blob/main/notebooks/Colab.ipynb`

## 1) Bootstrap — Drive + GitHub clone

In [ ]:
# Mount Drive and clone (or pull) the GitHub repo.
import os, subprocess
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = '/content/SeisMambaKAN'
REPO_URL = 'https://github.com/huseyinokanozturk/SeisMambaKAN.git'

if Path(REPO_DIR, '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
import sys; sys.path.insert(0, REPO_DIR)
print('cwd =', os.getcwd())

## 2) Setup — install deps + sync data

`--data-mode all` copies the full dataset (≈ few GB); use `--data-mode sample` for smoke tests.

In [ ]:
# Typer is needed before run.py can launch its own commands.
!pip install -q typer rich pyyaml tqdm
!python run.py setup --data-mode sample

## 3) Train / Eval / Infer

Pick one. All flags override values from `configs/config.yaml`.
Checkpoints are mirrored to Drive automatically (see `paths.yaml -> experiments.drive_root_dir`).

In [ ]:
# Project state at a glance
!python run.py status

In [ ]:
# Train (override anything from config.yaml on the CLI) for testing: --epochs 2 --batch-size 32 --data-mode sample
!python run.py train --epochs 20 --batch-size 32 --data-mode sample

In [ ]:
# Evaluate the latest experiment on val (auto-picks exp_NNN)
!python run.py eval --split val

In [ ]:
# Plot inference on a single trace (latest exp, random index)
!python run.py infer --split test

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir experiments --port 6006